# Getting Started
This tutorial demonstrates the configuration and use of a simple BSK-RL environment.
BSK-RL and dependencies should already be installed at this point (see [Installation](../install.rst)
if you haven't installed the package yet).

## Load Modules
In this tutorial, the environment will be created with `gym.make`, so it is necessary to
import the top-level `bsk_rl` module as well as `gym` and `bsk_rl` components.

In [11]:
# %matplotlib qt  # Uncomment to use interactive plotting, may need `pip install PyQt5`

import numpy as np
from functools import partial
from bsk_rl import act, obs, sats, ConstellationTasking, comm
from bsk_rl.sim import dyn, fsw
from bsk_rl.utils.orbital import relative_to_chief, random_orbit
from bsk_rl.scene import FibonacciSphereRSOPoints
from bsk_rl.data import RSOInspectionReward

from Basilisk.architecture import bskLogging
from Basilisk.utilities.RigidBodyKinematics import MRP2C

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)


If no errors were raised, you have a functional installation of `bsk_rl`.

## Configure the Satellite
[Satellites](../api_reference/sats/index.rst) are configurable agents in the environment.
To make a new environment, start by specifying the [observations](../api_reference/obs/index.rst)
and [actions](../api_reference/act/index.rst) of a satellite type, as well as the underlying
Basilisk [simulation](../api_reference/sim/index.rst) models used by the satellite.

In [12]:
import types


class TumbleSat(sats.Satellite):
    observation_spec = [
        obs.SatProperties(dict(prop="r_BN_N"), dict(prop="sigma_BN")),
    ]
    action_spec = [act.Drift()]
    dyn_type = types.new_class("Dyn", (dyn.ConjunctionDynModel, dyn.RSODynModel))
    fsw_type = fsw.BasicFSWModel


class ThrustSat(sats.Satellite):
    observation_spec = [
        obs.SatProperties(
            dict(prop="r_BN_N"),
            dict(prop="c_hat_N"),
            dict(prop="storage_level_fraction"),
        ),
        obs.RelativeProperties(
            dict(prop="r_DC_N"),
            chief_name="Tumbler",
        ),
    ]
    action_spec = [act.MagicThrust(max_dv=100, fsw_action="action_inspect_rso")]
    dyn_type = types.new_class("Dyn", (dyn.ConjunctionDynModel, dyn.RSOImagingDynModel))
    fsw_type = types.new_class(
        "FSW",
        (
            fsw.SteeringFSWModel,
            fsw.MagicOrbitalManeuverFSWModel,
            fsw.RSOImagingFSWModel,
        ),
    )


## Making the Environment
For this example, we will be using the single-agent [SatelliteTasking](../api_reference/index.rst) 
environment. Along with passing the satellite that we configured, the environment takes
a [scenario](../api_reference/scene/index.rst), which defines the environment the
satellite is acting in, and a [rewarder](../api_reference/data/index.rst), which defines
how data collected from the scenario is rewarded.

In [13]:
scanner_sat_args = dict(
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.1,
    instrumentBaudRate=1,
    dataStorageCapacity=1e6,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
)

env = ConstellationTasking(
    satellites=[
        TumbleSat(
            "Tumbler",
            obs_type=dict,
            sat_args=dict(
                # sigma_init=np.zeros(3),
                # omega_init=np.zeros(3),
            ),
        ),
        ThrustSat("Thrust-1", obs_type=dict, sat_args=scanner_sat_args),
        # ThrustSat("Thrust-2", obs_type=dict, sat_args=scanner_sat_args),
    ],
    sat_arg_randomizer=relative_to_chief(
        chief_name="Tumbler",
        chief_orbit=partial(random_orbit, i=0, Omega=0, omega=0, f=0),
        deputy_relative_state={
            "Thrust-1": np.array([50, 0, 0, 0, 0, 0]),
            # "Thrust-2": np.array([-50, 0, 0, 0, 0, 0]),
        },
    ),
    scenario=FibonacciSphereRSOPoints(
        n_points=500,
        radius=10,
        theta_min=np.radians(30),
    ),
    # communicator=comm.LOSCommunication(),
    rewarder=RSOInspectionReward(),
    time_limit=5700.0 * 3,
    log_level="INFO",
)
env.action_spaces

2024-12-12 10:59:43,536                                WARNING    Creating logger for new env on PID=54499. Old environments in process may now log times incorrectly.


{'Tumbler': Discrete(1),
 'Thrust-1': Box([-100. -100. -100.    0.], [100. 100. 100.  inf], (4,), float32)}

## Interacting with the Environment

First, the environment is reset.

In [14]:
observation, info = env.reset()  # seed=0)

2024-12-12 10:59:43,854 gym                            INFO       Resetting environment with seed=281930216
2024-12-12 10:59:43,970 sats.satellite.Thrust-1.FSW    INFO       <0.00> Thrust-1: <Basilisk.architecture.messaging.NavTransMsgPayload.NavTransMsg_C; proxy of <Swig Object of type 'NavTransMsg_C *' at 0x32e667330> >
2024-12-12 10:59:44,043 gym                            INFO       <0.00> Environment reset


Next, we take the scan action (`action=0`) a few times. This allows for the satellite to
settle its attitude in the nadir pointing mode to satisfy imaging conditions. Note that 
the logs show little or no data accumulated in the first two steps as it settles, but
achieves 60 reward (corresponding to 60 seconds of imaging) by the third step.

In [ ]:
for _ in range(1):
    actions = {
        "Thrust-1": np.array([0, 0, 0, 1000]),
        # "Thrust-2": np.array([0, 0, 0, 1]),
    }
    # for sat in env.satellites:
    #     if sat.requires_retasking:
    #         if isinstance(sat, TumbleSat):
    #             actions[sat.name] = 0
    #         else:
    #             actions[sat.name] = np.concatenate(
    #                 (np.random.uniform(-20, 20, 3), np.random.uniform(0, 200, 1))
    #             )

    observation, reward, terminated, truncated, info = env.step(actions)

    # print(env.satellites[1].data_store.data.point_inspect_status)
    # print("storage:", observation["Thrust-1"]["sat_props"]["storage_level_fraction"])

BN = MRP2C(observation["Tumbler"]["sat_props"]["sigma_BN"])

fig, ax = plt.subplots(1, 1, subplot_kw=dict(projection="3d"))
for point, inspected in env.satellites[1].data_store.data.point_inspect_status.items():
    # print(point, inspected)
    ax.scatter(*(BN.T @ point.r_PB_B), color="tab:green" if inspected else "tab:red")

# ax.scatter(
#     *observation["Thrust-1"]["rel_props"]["r_DC_N"], color="tab:blue", marker="x"
# )
ax.set_aspect("equal")

2024-12-12 10:59:44,049 gym                            INFO       <0.00> === STARTING STEP ===
2024-12-12 10:59:44,050 sats.satellite.Tumbler         WARNING    <0.00> Tumbler: Requires retasking but received no task.
2024-12-12 10:59:44,051 sats.satellite.Thrust-1        INFO       <0.00> Thrust-1: Thrusting with inertial dV [0 0 0] with 1000 second drift.
2024-12-12 10:59:44,052 sats.satellite.Thrust-1        INFO       <0.00> Thrust-1: setting timed terminal event at 1000.0
2024-12-12 10:59:44,052 sats.satellite.Thrust-1        INFO       <0.00> Thrust-1: FSW action action_inspect_rso activated.
2024-12-12 10:59:44,311 sats.satellite.Thrust-1        INFO       <1000.00> Thrust-1: timed termination at 1000.0 
2024-12-12 10:59:44,959 sats.satellite.Thrust-1        INFO       <1000.00> Thrust-1: Inspected 44 points this step
2024-12-12 10:59:44,959 data.base                      INFO       <1000.00> Data reward: {}
2024-12-12 10:59:44,959 sats.satellite.Tumbler         INFO       <1000

[ 3.05834745e+06  6.14678990e+06 -3.30911937e+01]
[ 3.05846436e+06  6.14688063e+06 -3.30899829e+01]
